In [2]:
import os, json, math
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

In [9]:
# ------------------------------------------------------------
# Pathway mining + adjacency + weights for 80 DE genes (LCD)
# Requirements:
#   pip install gprofiler-official pandas numpy networkx scikit-learn
# Inputs:
#   /mnt/data/80_DESeq2_significant_genes_lfc_fdr.csv
#     expected columns (case-insensitive): gene | log2fc | padj
# Outputs (in ./results/):
#   gprofiler_enrichment.csv
#   pathways_dict.json
#   gene_pathway_incidence.csv  (genes x pathways, 0/1)
#   gene_adjacency_jaccard.csv  (genes x genes, float)
#   gene_weights.csv            (gene, de_weight, centrality, final_weight)
# ------------------------------------------------------------
import os, json, math
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

# ---- Config
IN_DE = Path("../results/80_DESeq2_significant_genes_lfc_fdr.csv")
OUTDIR = Path("../results/pathway"); OUTDIR.mkdir(exist_ok=True, parents=True)
ORGANISM = "hsapiens"
SOURCES = ["KEGG","REAC","GO:BP","WP"]   # KEGG, Reactome, GO:BP, WikiPathways
ALPHA, BETA = 1.0, 1.0                   # weights for DE vs centrality (agent can tune)
P_ADJ_MAX = 0.10                         # keep enriched terms with adj p <= 0.1 (looser for coverage)

# ---- Helpers
def _std_col(df, names):
    for c in df.columns:
        if c.lower() in names: return c
    raise ValueError(f"Could not find any of {names} in columns: {list(df.columns)}")

def rank_norm(s: pd.Series) -> pd.Series:
    r = s.rank(method="average")
    return (r - r.min()) / (r.max() - r.min() + 1e-12)

# ---- 1) Load DE genes (80)
de_raw = pd.read_csv(IN_DE)
gene_col = _std_col(de_raw, {"gene","genes","symbol","gene_symbol"})
lfc_col  = _std_col(de_raw, {"log2fc","log2_fold_change","log2foldchange","logfc"})
padj_col = _std_col(de_raw, {"padj","fdr","adj_p","adj_pval","padj_bh"})

de = de_raw.rename(columns={gene_col:"gene", lfc_col:"log2FC", padj_col:"padj"})
de["gene"] = de["gene"].astype(str)
gene_list = de["gene"].dropna().unique().tolist()
if len(gene_list) == 0:
    raise ValueError("No genes found in input!")

# ---- 2) g:Profiler enrichment
# NOTE: This calls the public g:Profiler API; requires internet.
from gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)
enr = gp.profile(
    organism=ORGANISM,
    query=gene_list,
    sources=SOURCES,
    user_threshold=P_ADJ_MAX,
    no_evidences=False
)
# Keep only terms with intersections (genes)
enr = enr[enr["intersections"].notna()].copy()
enr.to_csv(OUTDIR / "gprofiler_enrichment.csv", index=False)

# ---- 3) Build pathway dictionary and incidence matrix
# g:Profiler 'intersections' are the overlapping genes from your list (comma-sep)
def parse_genes(x):
    if isinstance(x, str):
        return set([g.strip() for g in x.split(",") if g.strip()])
    return set()

# --- robust column picking for IDs/names across g:Profiler versions ---
cols = {c.lower(): c for c in enr.columns}  # case-insensitive

# possible id columns in different lib versions
id_candidates = ["term_id", "native", "id", "term", "term_code"]
name_candidates = ["name", "term_name", "description", "term_description", "term_name_with_source"]

id_col = next((cols[c] for c in id_candidates if c in cols), None)
name_col = next((cols[c] for c in name_candidates if c in cols), None)

# parse intersections (can be comma-separated string OR list)
def to_set(x):
    if isinstance(x, str):
        return set(g.strip() for g in x.split(",") if g.strip())
    if isinstance(x, (list, tuple, set)):
        return set(map(str, x))
    return set()

int_col = cols.get("intersections", None)
if int_col is None:
    # some older versions store per-query columns like 'query_0', etc.; fallback
    q_cols = [c for c in enr.columns if c.lower().startswith("query_")]
    if q_cols:
        enr["genes_in_term"] = enr[q_cols].apply(
            lambda row: set().union(*(to_set(row[c]) for c in q_cols)), axis=1
        )
    else:
        raise ValueError("No 'intersections' (or query_*) columns found in g:Profiler result.")
else:
    enr["genes_in_term"] = enr[int_col].apply(to_set)

# build pathway_id and pathway_name safely
if id_col is not None:
    enr["pathway_id"] = enr["source"].astype(str) + ":" + enr[id_col].astype(str)
else:
    # fallback: synthesize a stable id from source+name or row index
    if name_col is not None:
        enr["pathway_id"] = enr["source"].astype(str) + ":" + enr[name_col].astype(str)
    else:
        enr["pathway_id"] = enr["source"].astype(str) + ":row" + enr.index.astype(str)

if name_col is not None:
    enr["pathway_name"] = enr[name_col].astype(str)
else:
    enr["pathway_name"] = enr["pathway_id"]

# keep only terms with ≥2 genes from your list
enr = enr[enr["genes_in_term"].apply(lambda s: isinstance(s, set) and len(s) >= 2)]


# pathway -> set(genes)
pathways = {row["pathway_id"]+"|"+row["pathway_name"]: row["genes_in_term"]
            for _, row in enr.iterrows()
            if len(row["genes_in_term"]) >= 2}   # keep terms with >=2 genes

# save pathways dict
with open(OUTDIR / "pathways_dict.json","w") as f:
    json.dump({k: sorted(list(v)) for k,v in pathways.items()}, f, indent=2)

# Construct gene x pathway incidence for your genes only
genes = sorted(set(gene_list))
pw_names = list(pathways.keys())
inc = pd.DataFrame(0, index=genes, columns=pw_names, dtype=int)
for p, gset in pathways.items():
    present = sorted(gset & set(genes))
    if present:
        inc.loc[present, p] = 1
inc.to_csv(OUTDIR / "gene_pathway_incidence.csv")

# ---- 4) Build gene–gene adjacency by shared pathways (Jaccard of memberships)
def jaccard(a: np.ndarray, b: np.ndarray) -> float:
    inter = int((a & b).sum())
    union = int((a | b).sum())
    return inter / union if union > 0 else 0.0

inc_bool = inc.astype(bool).to_numpy()
gidx = inc.index.tolist()
A = np.zeros((len(gidx), len(gidx)), dtype=float)
for i in range(len(gidx)):
    Ai = inc_bool[i]
    for j in range(i+1, len(gidx)):
        Aj = inc_bool[j]
        Aij = jaccard(Ai, Aj)
        if Aij>0:
            A[i,j] = Aij; A[j,i] = Aij

adj = pd.DataFrame(A, index=gidx, columns=gidx)
adj.to_csv(OUTDIR / "gene_adjacency_jaccard.csv")

# ---- 5) Centrality (degree = sum of weights), DE weight, and final weight
deg = pd.Series(A.sum(axis=1), index=gidx, name="centrality")

# DE weight = |LFC| * -log10(padj), clipped for safety
de2 = de.set_index("gene").reindex(gidx)
de_strength = (de2["log2FC"].abs() * (-np.log10(np.clip(de2["padj"], 1e-300, 1)))).fillna(0.0)
de_strength.name = "de_weight"

w = ALPHA*rank_norm(de_strength) + BETA*rank_norm(deg)
# rescale to (0,1] to avoid zeros
w = (w - w.min())/(w.max() - w.min() + 1e-12) + 1e-6

# Create weights dataframe with gene symbols as index
weights = pd.concat([de_strength, deg, w.rename("final_weight")], axis=1)

# Add gene symbol and gene ID columns to weights for easier mapping later
weights = weights.reset_index().rename(columns={'index': 'gene_symbol'})

# Load gene annotation to add gene IDs
df_gene_annot = pd.read_csv('../data/Human.GRCh38.p13.annot.tsv', sep='\t', index_col=None, header=0, low_memory=False)

# Find symbol and ID columns in annotation
symbol_col = None
id_col = None

for col in df_gene_annot.columns:
    col_lower = col.lower()
    if 'symbol' in col_lower or 'gene_name' in col_lower:
        symbol_col = col
    elif 'gene_id' in col_lower or 'ensembl' in col_lower or col_lower in ['geneid', 'id']:
        id_col = col

if symbol_col is None or id_col is None:
    # Try common column patterns
    possible_symbol_cols = [c for c in df_gene_annot.columns if any(x in c.lower() for x in ['symbol', 'name', 'gene'])]
    possible_id_cols = [c for c in df_gene_annot.columns if any(x in c.lower() for x in ['id', 'ensembl'])]
    
    if possible_symbol_cols:
        symbol_col = possible_symbol_cols[0]
    if possible_id_cols:
        id_col = possible_id_cols[0]

# Add gene_id column to weights if annotation is available
if symbol_col is not None and id_col is not None:
    gene_mapping = df_gene_annot.set_index(symbol_col)[id_col].dropna().to_dict()
    weights['gene_id'] = weights['gene_symbol'].map(gene_mapping)
    print(f"Added gene IDs to {weights['gene_id'].notna().sum()} out of {len(weights)} genes")
else:
    weights['gene_id'] = None
    print("Could not find gene annotation columns - gene_id will be None")

# Set gene_symbol back as index for backward compatibility
weights = weights.set_index('gene_symbol')

weights.to_csv(OUTDIR / "gene_weights.csv")

print("Done:")
print(f"  {len(genes)} genes")
print(f"  {inc.shape[1]} enriched pathways kept (padj <= {P_ADJ_MAX})")
print("Saved:")
print(f"  {OUTDIR/'gprofiler_enrichment.csv'}")
print(f"  {OUTDIR/'pathways_dict.json'}")
print(f"  {OUTDIR/'gene_pathway_incidence.csv'}")
print(f"  {OUTDIR/'gene_adjacency_jaccard.csv'}")
print(f"  {OUTDIR/'gene_weights.csv'}")

# -------- Optional: weighted PCA if you already have VST expression --------
# If you have an expression matrix `expr_vst` (samples x genes, columns = gidx),
# you can uncomment this block to produce weighted PCs immediately.
"""
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# expr_vst = pd.read_csv("/path/to/your_vst_matrix.csv", index_col=0)
expr_vst = expr_vst.reindex(columns=gidx)   # align columns
X = expr_vst.values
Xz = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

sqrtw = np.sqrt(weights.loc[gidx, "final_weight"].values)
Xw = Xz * sqrtw[None, :]

pca = PCA(n_components=100, svd_solver="full", random_state=0)
Z = pca.fit_transform(Xw)
pd.DataFrame(Z, index=expr_vst.index).to_csv(OUTDIR / "weighted_pca_scores.csv")
pd.DataFrame(pca.components_, columns=gidx).to_csv(OUTDIR / "weighted_pca_loadings.csv")
"""


Found gene ID column: GeneID
Added gene IDs to 80 out of 80 genes using DE data


Found gene ID column: GeneID
Added gene IDs to 80 out of 80 genes using DE data


KeyError: 'gene_symbol'

In [8]:
weights

,gene_symbol,de_weight,centrality,final_weight
gene_id,,,,
ENSG00000188984,AADACL3,22.019044,0.000000,0.350319
ENSG00000109107,ALDOC,75.843763,1.898291,0.871143
ENSG00000165566,AMER2,1.723972,0.000000,0.012740
ENSG00000130173,ANGPTL8,39.262223,0.000000,0.420383
ENSG00000171885,AQP4,4.185989,0.000000,0.089173
ENSG00000118276,B4GALT6,31.683718,0.000000,0.407644
ENSG00000243449,C4orf48,29.735380,0.000000,0.388536
ENSG00000178722,C5orf64,48.814289,0.000000,0.452230
ENSG00000113600,C9,4.158515,0.000000,0.082804


In [14]:
df_gene_annot = pd.read_csv('../data/Human.GRCh38.p13.annot.tsv', sep='\t', index_col = None, header=0, low_memory=False)
df_gene_annot

,GeneID,Symbol,Description,Synonyms,GeneType,EnsemblGeneID,Status,ChrAcc,ChrStart,ChrStop,Orientation,Length,GOFunctionID,GOProcessID,GOComponentID,GOFunction,GOProcess,GOComponent
0,100287102,DDX11L1,DEAD/H-box helicase 11 like 1 (pseudogene),NaN,pseudo,ENSG00000290825,active,NC_000001.11,11874,14409,positive,1652,NaN,NaN,NaN,NaN,NaN,NaN
1,653635,WASH7P,"WASP family homolog 7, pseudogene",FAM39F|WASH5P,pseudo,NaN,active,NC_000001.11,14362,29370,negative,1769,NaN,NaN,NaN,NaN,NaN,NaN
2,102466751,MIR6859-1,microRNA 6859-1,hsa-mir-6859-1,ncRNA,ENSG00000278267,active,NC_000001.11,17369,17436,negative,68,NaN,NaN,NaN,NaN,NaN,NaN
3,107985730,MIR1302-2HG,MIR1302-2 host gene,NaN,ncRNA,NaN,active,NC_000001.11,29926,31295,positive,538,NaN,NaN,NaN,NaN,NaN,NaN
4,100302278,MIR1302-2,microRNA 1302-2,MIRN1302-2|hsa-mir-1302-2,ncRNA,ENSG00000284332,active,NC_000001.11,30366,30503,positive,138,NaN,GO:0035195,NaN,NaN,miRNA-mediated gene silencing,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39371,4541,ND6,NADH dehydrogenase subunit 6,MTND6,protein-coding,NaN,active,NC_012920.1,14149,14673,negative,525,GO:0008137,GO:0006120///GO:0009060///GO:0032981///GO:0035...,GO:0005739///GO:0005743///GO:0005747,NADH dehydrogenase (ubiquinone) activity,"mitochondrial electron transport, NADH to ubiq...",mitochondrion///mitochondrial inner membrane//...
39372,4556,TRNE,tRNA-Glu,MTTE,tRNA,NaN,active,NC_012920.1,14674,14742,negative,69,NaN,NaN,NaN,NaN,NaN,NaN
39373,4519,CYTB,cytochrome b,MTCYB,protein-coding,NaN,active,NC_012920.1,14747,15887,positive,1141,GO:0008121///GO:0046872,GO:0006122///GO:0045333///GO:1902600,GO:0005739///GO:0005743///GO:0005750///GO:0016020,ubiquinol-cytochrome-c reductase activity///me...,"mitochondrial electron transport, ubiquinol to...",mitochondrion///mitochondrial inner membrane//...
39374,4576,TRNT,tRNA-Thr,MTTT,tRNA,NaN,active,NC_012920.1,15888,15953,positive,66,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
expr_vst

,GSM2520157,GSM2520158,GSM2520159,GSM2520160,GSM2520161,GSM2520162,GSM2520163,GSM2520164,GSM2520165,GSM2520166,...,GSM2520529,GSM2520530,GSM2520531,GSM2520532,GSM2520533,GSM2520534,GSM2520535,GSM2520536,GSM2520537,GSM2520538
GeneID,,,,,,,,,,,,,,,,,,,,,
100287102,21.005022,2.253288,9.606387,1.787645,2.985084,18.561131,4.279128,1.987068,7.709543,2.399610,...,1.001034,6.444927,6.151350,4.607232,5.725350,4.194643,2.588963,2.837753,7.136887,4.514742
653635,573.437111,482.203642,599.089232,362.891850,396.269921,491.324046,491.029974,419.271322,651.105935,295.951870,...,301.311342,479.789033,343.108653,310.220309,350.200585,442.954275,325.346371,303.639540,202.687600,558.323045
102466751,13.303181,11.266440,19.212774,8.938223,12.686608,13.101975,14.976949,14.903009,20.325158,9.598439,...,8.008275,20.766988,13.669667,6.910848,12.404925,24.328928,5.177926,8.513258,7.136887,15.049139
107985730,0.700167,0.000000,0.873308,0.000000,0.746271,1.091831,0.000000,0.993534,0.700868,0.799870,...,0.000000,0.716103,0.683483,1.535744,0.954225,0.838929,0.000000,0.945918,1.427377,0.752457
100302278,0.000000,0.000000,0.873308,0.000000,0.000000,0.000000,0.000000,0.000000,0.700868,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.945918,1.427377,0.752457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4541,65218.493984,53285.755684,86956.142713,88890.626715,58519.589752,93704.230291,96074.988846,86339.092872,130099.236104,81206.793640,...,107583.164659,81885.665935,88575.344297,116805.625222,114593.837604,164748.788991,112876.205083,124964.224368,104541.125575,67572.137611
4556,5002.696163,3511.749418,7082.527216,7030.806131,4890.314115,7153.678115,7319.448977,5973.126044,9628.518126,6954.868955,...,8242.516914,6163.498821,7023.475145,8175.533749,8248.321125,11225.702951,7444.995078,9371.205374,6320.427417,6075.337287
4519,177973.454113,166229.565643,227802.370435,242588.732369,135318.342320,270049.165182,262076.283209,247025.323967,322034.612635,261377.493147,...,234232.029724,289490.380672,215474.968310,282656.006345,274298.663315,397946.597448,257454.264135,314770.152301,286750.141182,200003.053127


Loading expression data and mapping gene identifiers...
Expression matrix shape: (39376, 382)
Expression matrix columns (first 5): ['GSM2520157', 'GSM2520158', 'GSM2520159', 'GSM2520160', 'GSM2520161']
Expression matrix index (first 5): [100287102, 653635, 102466751, 107985730, 100302278]
Expression matrix index type: <class 'numpy.int64'>
Weights index (gene symbols, first 5): ['AADACL3', 'ALDOC', 'AMER2', 'ANGPTL8', 'AQP4']
Weights has gene_id column: True
Genes with gene IDs: 80 out of 80
Expression matrix appears to have samples as columns - checking index for gene IDs
Common genes (string comparison): 80
Common genes (int comparison): 80
Using string comparison - found 80 common gene IDs
Using 80 matched genes for weighted PCA
Expression matrix shape: (39376, 382)
Expression matrix columns (first 5): ['GSM2520157', 'GSM2520158', 'GSM2520159', 'GSM2520160', 'GSM2520161']
Expression matrix index (first 5): [100287102, 653635, 102466751, 107985730, 100302278]
Expression matrix index 

Loading expression data and mapping gene identifiers...
Expression matrix shape: (39376, 382)
Expression matrix columns (first 5): ['GSM2520157', 'GSM2520158', 'GSM2520159', 'GSM2520160', 'GSM2520161']
Expression matrix index (first 5): [100287102, 653635, 102466751, 107985730, 100302278]
Expression matrix index type: <class 'numpy.int64'>
Weights index (gene symbols, first 5): ['AADACL3', 'ALDOC', 'AMER2', 'ANGPTL8', 'AQP4']
Weights has gene_id column: True
Genes with gene IDs: 80 out of 80
Expression matrix appears to have samples as columns - checking index for gene IDs
Common genes (string comparison): 80
Common genes (int comparison): 80
Using string comparison - found 80 common gene IDs
Using 80 matched genes for weighted PCA
Expression matrix shape: (39376, 382)
Expression matrix columns (first 5): ['GSM2520157', 'GSM2520158', 'GSM2520159', 'GSM2520160', 'GSM2520161']
Expression matrix index (first 5): [100287102, 653635, 102466751, 107985730, 100302278]
Expression matrix index 

ValueError: operands could not be broadcast together with shapes (80,382) (1,80) 

,GSM2520157,GSM2520158,GSM2520159,GSM2520160,GSM2520161,GSM2520162,GSM2520163,GSM2520164,GSM2520165,GSM2520166,...,GSM2520529,GSM2520530,GSM2520531,GSM2520532,GSM2520533,GSM2520534,GSM2520535,GSM2520536,GSM2520537,GSM2520538
GeneID,,,,,,,,,,,,,,,,,,,,,
100287102,21.005022,2.253288,9.606387,1.787645,2.985084,18.561131,4.279128,1.987068,7.709543,2.399610,...,1.001034,6.444927,6.151350,4.607232,5.725350,4.194643,2.588963,2.837753,7.136887,4.514742
653635,573.437111,482.203642,599.089232,362.891850,396.269921,491.324046,491.029974,419.271322,651.105935,295.951870,...,301.311342,479.789033,343.108653,310.220309,350.200585,442.954275,325.346371,303.639540,202.687600,558.323045
102466751,13.303181,11.266440,19.212774,8.938223,12.686608,13.101975,14.976949,14.903009,20.325158,9.598439,...,8.008275,20.766988,13.669667,6.910848,12.404925,24.328928,5.177926,8.513258,7.136887,15.049139
107985730,0.700167,0.000000,0.873308,0.000000,0.746271,1.091831,0.000000,0.993534,0.700868,0.799870,...,0.000000,0.716103,0.683483,1.535744,0.954225,0.838929,0.000000,0.945918,1.427377,0.752457
100302278,0.000000,0.000000,0.873308,0.000000,0.000000,0.000000,0.000000,0.000000,0.700868,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.945918,1.427377,0.752457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4541,65218.493984,53285.755684,86956.142713,88890.626715,58519.589752,93704.230291,96074.988846,86339.092872,130099.236104,81206.793640,...,107583.164659,81885.665935,88575.344297,116805.625222,114593.837604,164748.788991,112876.205083,124964.224368,104541.125575,67572.137611
4556,5002.696163,3511.749418,7082.527216,7030.806131,4890.314115,7153.678115,7319.448977,5973.126044,9628.518126,6954.868955,...,8242.516914,6163.498821,7023.475145,8175.533749,8248.321125,11225.702951,7444.995078,9371.205374,6320.427417,6075.337287
4519,177973.454113,166229.565643,227802.370435,242588.732369,135318.342320,270049.165182,262076.283209,247025.323967,322034.612635,261377.493147,...,234232.029724,289490.380672,215474.968310,282656.006345,274298.663315,397946.597448,257454.264135,314770.152301,286750.141182,200003.053127
